In [7]:
# --- make the transformer_pipeline package importable and resolve data paths ---
# Two anchors, found by walking up from the cwd (idempotent, so re-running this
# cell after os.chdir is safe):
#   * the package dir  -> the dir that contains transformer_pipeline/  (for imports)
#   * the data root    -> the dir that contains train/ AND embeddings/ (for chdir,
#     since the config's data paths are relative: "train/", "embeddings/")
import os, sys

def _walk_up(predicate):
    d = os.getcwd()
    while not predicate(d) and os.path.dirname(d) != d:
        d = os.path.dirname(d)
    return d if predicate(d) else None

_PKG_ROOT = _walk_up(lambda d: os.path.isdir(os.path.join(d, "transformer_pipeline")))
_DATA_ROOT = _walk_up(
    lambda d: os.path.isdir(os.path.join(d, "train"))
    and os.path.isdir(os.path.join(d, "embeddings"))
)
if _PKG_ROOT is None:
    raise RuntimeError("Could not locate the 'transformer_pipeline' package above the cwd.")
if _DATA_ROOT is None:
    raise RuntimeError("Could not locate a directory containing both 'train/' and 'embeddings/'.")

_PKG = os.path.join(_PKG_ROOT, "transformer_pipeline")
if _PKG not in sys.path:
    sys.path.insert(0, _PKG)
os.chdir(_DATA_ROOT)
print("cwd:", os.getcwd())

cwd: /Users/yairbendavid/Desktop/Transformer/multiomic-missing-data-pipeline


# Embedding ablation - under CLR (do the node2vec embeddings earn their place?)

Earlier (pre-CLR) the embedding ablations showed both models barely beat
mean-fill, so it was unclear the node2vec embeddings carried any signal. CLR
changed the input/target space and lifted both models, so this re-runs the
embedding-freeze ablation **in the CLR + scale setup** to confirm whether the
embeddings actually contribute now - especially for **Model B**, whose input
tokens ARE the metabolite embeddings (and where the 25 random-init rows live).

**The three conditions (what is learnable):**

```
all fixed           every embedding row FROZEN  (pure node2vec lookup, no learning)
77 fixed / 25 learn  calibrated metabolite rows frozen, the 25 random-init
                     (island/missing) metabolite rows TRAINABLE   (Model B only)
everything trainable ALL embedding rows trainable (both models)
```

How freezing works: each row keeps a frozen `embedding_base`; a zero-init
`embedding_delta` is added only on rows whose `trainable_mask=1`. So "frozen"
rows stay bit-for-bit equal to node2vec, and only the unmasked rows can move.

Note: the 25 random-init rows are **metabolites**, used as Model B's input
tokens. Model A's inputs are the microbe embeddings (no random rows), so for
Model A "all fixed" and "77 fixed / 25 learn" are identical - only "everything
trainable" differs for A.

**The bar (fair CLR ridge ceilings, per-feature r): A 0.15 | B 0.21.**
mean-fill per-feature r is ~0, so any clear positive r means the model is using
real structure. If "all fixed" already reaches the main-run r (~0.17 A / ~0.14
B), the *frozen* node2vec identities are doing the work. Flip `RANDOMIZE_EMB`
to True to re-run all three on RANDOM embeddings: if scores barely change, the
node2vec content is not what is helping (only the per-feature identity is).

Run from the repo root pattern (cwd must see `embeddings/` and `train/`):
Cell > Run All. This is 3 conditions x 2 models = 6 trainings.

In [9]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import copy, builtins
import numpy as np
import pandas as pd
import torch

from pipeline.config import Config
from pipeline.pipeline import ImputationPipeline
from pipeline.data import make_loader
from pipeline.evaluation import Evaluator, per_feature_pearson
from pipeline.models import ImputationTransformer
from pipeline.trainer import Trainer

EPOCHS  = 60
PERMS   = 99
VERBOSE = False          # True -> per-epoch training logs
RANDOMIZE_EMB = False    # True -> replace node2vec with random embeddings (P3 test)

# ---- one CLR + scale data prep, shared by every condition ----
cfg = Config()                       # microbiome_clr=True, tokenizer="scale" (defaults)
cfg.train.max_epochs = EPOCHS
cfg.train.patience   = EPOCHS        # no early stop -> full curves
cfg.eval.mantel_permutations = PERMS

pipe = ImputationPipeline(cfg)
art = pipe.prepare_data()

# CLR frames (microbiome) + standardized metabolome
micro_train = art.micro_train_proc           # CLR
micro_val   = art.micro_val_proc             # CLR
metab_train = art.metab_train_std
metab_val   = art.metab_val_std

micro_emb = art.micro_emb                    # (170, 128) node2vec
metab_emb = art.metab_emb                    # (102, 128) node2vec
metab_rand_mask = art.metab_trainable_mask.astype(bool)   # True on the 25 random rows

def random_like(emb, seed=0):
    rng = np.random.default_rng(seed)
    std = float(np.std(emb)) or 1.0
    return (rng.standard_normal(emb.shape) * std).astype(np.float32)

if RANDOMIZE_EMB:
    micro_emb = random_like(micro_emb, 0)
    metab_emb = random_like(metab_emb, 1)
    print("[emb] RANDOMIZED embeddings (node2vec discarded)")

n_metab = metab_train.shape[1]
n_micro = micro_train.shape[1]
print(f"[data] n_micro={n_micro} n_metab={n_metab} "
      f"random-init metab rows={int(metab_rand_mask.sum())}")

FileNotFoundError: [Errno 2] No such file or directory: 'train/microbiome.csv'

### Train each condition and score both models

Model A is selected on `val_mse` (honest). Model B uses an **identity** head
(CLR target) so its `val_mse` is meaningful and it is selected on `val_mse`
too. Per-feature r for Model B is measured in CLR space, directly comparable to
the ridge B-CLR ceiling (0.21).

In [ ]:
true_micro_val = micro_val.values            # CLR space
true_metab_val = metab_val.values

evaluator = Evaluator(cfg.eval)

def _quiet_fit(trainer, tl, vl, verbose):
    if verbose:
        trainer.fit(tl, vl); return
    real = builtins.print
    builtins.print = lambda *a, **k: None
    try:
        trainer.fit(tl, vl)
    finally:
        builtins.print = real

def train_one(model, name, Xtr, Ytr, Xva, Yva, eval_fn, pf_fn):
    tcfg = copy.deepcopy(cfg.train)
    tl = make_loader(Xtr, Ytr, tcfg.batch_size, True, 0)
    vl = make_loader(Xva, Yva, tcfg.batch_size, False, 0)
    tr = Trainer(model, tcfg, name=name, mantel_eval_fn=eval_fn, per_feature_fn=pf_fn)
    _quiet_fit(tr, tl, vl, VERBOSE)
    return tr

def selected(tr):
    e = tr.history.best_epoch
    rec = [r for r in tr.history.records if r.epoch == e][0]
    return rec.per_feature_r, rec.val_mse, rec.mantel

def peak_r(tr):
    vals = [r.per_feature_r for r in tr.history.records if r.per_feature_r is not None]
    return max(vals) if vals else float("nan")

# input -> model -> prediction helpers for the eval/per-feature callbacks
def _predict(m, X):
    m.eval()
    with torch.no_grad():
        dev = next(m.parameters()).device
        xb = torch.tensor(X, dtype=torch.float32, device=dev)
        return m(xb).cpu().numpy()

def eval_a(m):  return evaluator.evaluate_model_a(true_micro_val, true_metab_val, _predict(m, true_micro_val))
def pf_a(m):    return per_feature_pearson(true_metab_val, _predict(m, true_micro_val))
def eval_b(m):  return evaluator.evaluate_model_b(true_micro_val, true_metab_val, _predict(m, true_metab_val))
def pf_b(m):    return per_feature_pearson(true_micro_val, _predict(m, true_metab_val))

# mean-fill MSE references
mean_metab = np.repeat(metab_train.values.mean(0, keepdims=True), len(true_metab_val), 0)
mean_micro = np.repeat(micro_train.values.mean(0, keepdims=True), len(true_micro_val), 0)
mse_mean_a = float(np.mean((mean_metab - true_metab_val) ** 2))
mse_mean_b = float(np.mean((mean_micro - true_micro_val) ** 2))

# (label, model_A embedding mask, model_B embedding mask)
ones_micro = np.ones(micro_emb.shape[0], bool)
ones_metab = np.ones(metab_emb.shape[0], bool)
CONDITIONS = [
    ("all fixed",            None,        None),
    ("77 fixed / 25 learn",  None,        metab_rand_mask),
    ("everything trainable", ones_micro,  ones_metab),
]

rows = []
for label, a_mask, b_mask in CONDITIONS:
    print(f"\n========== {label} ==========")
    # ---- Model A: micro(CLR) -> metab ----
    cfgA = copy.deepcopy(cfg.model_a)               # scale tokenizer, identity head
    modelA = ImputationTransformer(micro_emb, n_metab, cfgA, trainable_mask=a_mask)
    trA = train_one(modelA, f"A[{label}]", micro_train, metab_train,
                    micro_val, metab_val, eval_a, pf_a)
    sa_r, sa_mse, sa_mant = selected(trA)
    predA = _predict(modelA, true_micro_val)
    rows.append({
        "condition": label, "model": "A micro->metab",
        "emb_trainable": modelA.tokenizer.n_trainable,
        "sel_r": round(sa_r, 4), "peak_r": round(peak_r(trA), 4),
        "sel_val_mse": round(sa_mse, 5),
        "mse_model": round(float(np.mean((predA - true_metab_val) ** 2)), 5),
        "mse_mean": round(mse_mean_a, 5),
        "mantel_r": round(sa_mant.statistic, 4) if sa_mant else None,
    })
    # ---- Model B: metab -> micro(CLR), identity head ----
    cfgB = copy.deepcopy(cfg.model_b)
    cfgB.output_activation = "identity"             # CLR target
    modelB = ImputationTransformer(metab_emb, n_micro, cfgB, trainable_mask=b_mask)
    trB = train_one(modelB, f"B[{label}]", metab_train, micro_train,
                    metab_val, micro_val, eval_b, pf_b)
    sb_r, sb_mse, sb_mant = selected(trB)
    predB = _predict(modelB, true_metab_val)
    rows.append({
        "condition": label, "model": "B metab->micro",
        "emb_trainable": modelB.tokenizer.n_trainable,
        "sel_r": round(sb_r, 4), "peak_r": round(peak_r(trB), 4),
        "sel_val_mse": round(sb_mse, 5),
        "mse_model": round(float(np.mean((predB - true_micro_val) ** 2)), 5),
        "mse_mean": round(mse_mean_b, 5),
        "mantel_r": round(sb_mant.statistic, 4) if sb_mant else None,
    })

ablation = pd.DataFrame(rows)

### Summary vs the fair CLR ridge ceilings

In [ ]:
print("FAIR CLR RIDGE CEILINGS (per-feature r):  A 0.15 | B 0.21")
print("mean-fill per-feature r ~ 0 (constant prediction)\n")
print(ablation.to_string(index=False))

print("\nBeats mean-fill (MSE)?")
for _, r in ablation.iterrows():
    tag = "BEATS" if r.mse_model < r.mse_mean else "no better than"
    print(f"  {r.model:16s} {r.condition:22s} {tag} mean-fill "
          f"(model={r.mse_model}, mean={r.mse_mean})")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
ceil = {"A micro->metab": 0.15, "B metab->micro": 0.21}
for ax, model_name in zip(axes, ["A micro->metab", "B metab->micro"]):
    sub = ablation[ablation.model == model_name]
    ax.bar(sub.condition, sub.sel_r, color=["#4c78a8", "#f58518", "#54a24b"])
    ax.axhline(ceil[model_name], ls="--", color="crimson",
               label=f"ridge ceiling {ceil[model_name]}")
    ax.set_title(model_name)
    ax.set_ylabel("selected per-feature r")
    ax.tick_params(axis="x", rotation=20)
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)
fig.suptitle("Embedding-freeze ablation under CLR + scale", y=1.03)
plt.tight_layout()
plt.show()

### How to read this

- **Do the embeddings earn their place?** Look at `all fixed` (no embedding
  learning at all). If it already reaches the main-run r (~0.17 A / ~0.14 B),
  the **frozen node2vec identities** are carrying the signal - the embeddings
  earn their place as fixed feature identities.
- **Does unfreezing help?** Compare `all fixed` -> `77 fixed / 25 learn`
  (Model B only) -> `everything trainable`. If `sel_r` does not rise (or `peak_r`
  rises while `sel_r`/`sel_val_mse` worsen), unfreezing is just adding capacity
  to overfit - keep everything frozen (the current pipeline default,
  `unfreeze_random_metabolites=False`).
- **node2vec vs random?** Re-run with `RANDOMIZE_EMB=True`. If the three
  conditions land at basically the same r as with node2vec, then node2vec's
  learned content is NOT what helps - only having a distinct per-feature vector
  matters, and a random table would do. If node2vec is clearly higher, the graph
  structure is contributing.
- `emb_trainable` confirms the mask: 0 = fully frozen, 25 = the random metab
  rows, 102/170 = all rows of that model's embedding table.
- Reminder: Model B's r is in CLR space (comparable to ridge B 0.21); mean-fill
  r is ~0, so clearing it is the minimum bar, and ridge is the real target.